# Análisis de resultados del banco final · Globix

Lee los resultados que genera `python evaluacion.py <banco> --repeticiones 3` y muestra:

1. Los 6 indicadores con su meta, el promedio, el rango entre corridas y el intervalo de confianza al 95 % de cada corrida.
2. El análisis de error por segmento (universo de cotización, universo de conversación y tipo de pedido de productos).
3. La comparación con el baseline de reglas (sin LLM).

**Requisito:** haber corrido `evaluacion.py`; los resultados quedan en `evaluacion/reportes/banco_final/<corrida>/`.

In [ ]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 80)
RAIZ = Path.cwd() if (Path.cwd() / "evaluacion.py").exists() else Path.cwd().parent
REPORTES = RAIZ / "evaluacion" / "reportes" / "banco_final"

# Corrida combinada más reciente (<corrida>-xN); si no hay, la corrida simple más reciente
combinadas = sorted(REPORTES.glob("*-x*/resultado.json"), key=lambda p: p.stat().st_mtime)
simples = sorted(REPORTES.glob("*/resultado.json"), key=lambda p: p.stat().st_mtime)
archivo = (combinadas or simples)[-1]
resultado = json.loads(archivo.read_text(encoding="utf-8"))
meta = resultado["meta"]
print(f"Resultados: {archivo.parent.name}")
print(f"Casos ejecutados: {meta['ejecutados']} de {meta['total']} · errores: {meta['errores']} · juez: {meta['modelo_juez']}")

## 1. Indicadores frente a su meta

In [ ]:
filas = []
for k in resultado["kpis"]:
    filas.append({
        "Indicador": k["indicador"],
        "Por corrida (%)": " · ".join("—" if v is None else f"{v:.1f}" for v in k.get("por_corrida", [k["porcentaje"]])),
        "Promedio (%)": k["porcentaje"],
        "Mín (%)": k.get("minimo", k["porcentaje"]),
        "Máx (%)": k.get("maximo", k["porcentaje"]),
        "Meta (%)": k["meta"],
        "Cumple": "Sí" if k["cumple"] else "No",
    })
pd.DataFrame(filas)

### Intervalos de confianza por corrida

Clopper-Pearson al 95 % sobre los aciertos de cada corrida (la recopilación es un promedio por conversación y no tiene intervalo binomial).

In [ ]:
corridas = meta.get("corridas") or [archivo.parent.name]
filas = []
for nombre in corridas:
    r = json.loads((REPORTES / nombre / "resultado.json").read_text(encoding="utf-8"))
    for k in r["kpis"]:
        filas.append({"Corrida": nombre, "Indicador": k["indicador"],
                      "Aciertos": k.get("aciertos"), "Casos": k.get("casos"),
                      "%": k["porcentaje"], "IC 95 %": "—" if not k.get("ic95") else f"{k['ic95'][0]}–{k['ic95'][1]}"})
pd.DataFrame(filas).pivot(index="Indicador", columns="Corrida", values="IC 95 %")

## 2. Análisis de error por segmento

Aciertos sumados de todas las corridas por segmento de cada banco: universo de cotización (U01–U08), universo de conversación (CH01–CH06) y tipo de pedido de productos (con condiciones o no cotizable).

In [ ]:
seg = pd.DataFrame([{
    "Banco": s["banco"], "Segmento": s["segmento"], "Aciertos": s["aciertos"], "Casos": s["casos"],
    "%": s["porcentaje"], "IC 95 %": f"{s['ic95'][0]}–{s['ic95'][1]}" if s.get("ic95") else "—",
    "Casos con falla": ", ".join(sorted(set(s["fallas"]))) or "—",
} for s in resultado.get("segmentos", [])])
seg.sort_values(["Banco", "%"])

In [ ]:
# Segmentos más débiles (por debajo del 95 %)
seg[seg["%"] < 95].sort_values("%")

## 3. Comparación con el baseline de reglas

El baseline (`evaluacion/baseline_reglas.py`) es el mismo sistema con el LLM reemplazado por reglas de palabras clave. Sobre el banco final se corre con `python evaluacion/baseline_banco_final.py <banco> --comparar <corrida de Globix>` (mismo juez y mismas métricas; queda en `BASE-REGLAS/comparacion.json`). Debajo, además, las comparaciones anteriores sobre bancos de consultas reales.

In [ ]:
comparacion = REPORTES / "BASE-REGLAS" / "comparacion.json"
if comparacion.exists():
    banco_final_vs_reglas = pd.DataFrame(json.loads(comparacion.read_text(encoding="utf-8")))
    banco_final_vs_reglas = banco_final_vs_reglas.rename(columns={
        "indicador": "Indicador", "globix": "Globix (%)", "reglas": "Reglas (%)", "diferencia_pp": "Diferencia (pp)",
        "meta": "Meta (%)", "reglas_cumple": "Reglas cumple"})
    display(banco_final_vs_reglas)

baseline = pd.DataFrame([
    {"Conjunto": "Banco de 259 consultas reales y variantes", "Medida": "Éxito de tarea por checks (5 corridas)",
     "Globix (%)": 72.9, "Rango Globix": "71.4–73.7", "Reglas (%)": 60.6},
    {"Conjunto": "Banco de 259 consultas reales y variantes", "Medida": "Atención correcta según el juez (258 casos, 4 corridas)",
     "Globix (%)": 57.9, "Rango Globix": "57.4–59.3", "Reglas (%)": 37.2},
    {"Conjunto": "Lote de 60 casos reales", "Medida": "Atención correcta según el juez (54 evaluables, 3 corridas)",
     "Globix (%)": 76.5, "Rango Globix": "74.1–77.8", "Reglas (%)": 57.4},
])
baseline["Diferencia (pp)"] = (baseline["Globix (%)"] - baseline["Reglas (%)"]).round(1)
baseline

## 4. Tiempo de respuesta

In [ ]:
t = resultado.get("tiempos") or {}
print(f"Tiempo de procesamiento por turno: p50 {t.get('p50_s')} s · p95 {t.get('p95_s')} s ({t.get('turnos')} turnos)")
print("No incluye los 5 s de agrupación de mensajes de WhatsApp.")